- uncomment to install library

In [ ]:
#! pip install pydot

In [2]:
#! pip install graphviz 

## Setup Experiment

In [3]:
labels = ['AF', 'N']

fs = 250

feature_type = "vgg16" # ubah ke 'vgg16', 'resnet50', 'densenet121'

threshold_acc = 0.98

In [4]:
import os
import datetime

experiment_folder = "experiment/"

experiment_name = feature_type + "_" + datetime.datetime.now().strftime("%Y_%m_%d__%H_%M_%S")

if not os.path.exists(experiment_folder + experiment_name) :
    os.mkdir(experiment_folder + experiment_name)

In [5]:

dataset_folder = 'dataset/'
filenames = []
for filename in os.listdir(dataset_folder):
    if filename.find(".npz") > -1 or filename.find("_all") > -1:
        filenames.append(filename)

In [6]:
filenames

['test_all.csv',
 'test_all_Conv_AE.csv',
 'test_densenet121_feature.npz',
 'test_resnet50_feature.npz',
 'test_vgg16_feature.npz',
 'train_all.csv',
 'train_all_Conv_AE.csv',
 'train_vgg16_feature.npz',
 'train_vgg16_feature_flatten.npz']

In [7]:

import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from keras.utils.np_utils import to_categorical

Using TensorFlow backend.


## Load dataset

- load base data

In [8]:
train_Conv_AE_df = pd.read_csv(dataset_folder + "train_all_Conv_AE.csv", header=None)
test_Conv_AE_df = pd.read_csv(dataset_folder + "test_all_Conv_AE.csv", header=None)

In [9]:
y_train = train_Conv_AE_df.iloc[:, 600].values
y_test = test_Conv_AE_df.iloc[:, 600].values

y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

- load feature data

In [10]:
X_train = np.load(dataset_folder + 'train_vgg16_feature.npz')['vgg16_feature']
X_test = np.load(dataset_folder + 'test_vgg16_feature.npz')['vgg16_feature']

In [11]:
X_train = np.squeeze(X_train)
X_test = np.squeeze(X_test)

In [12]:
y_train.shape, X_train.shape, y_test.shape, X_test.shape

((32620, 2), (32620, 7, 7, 512), (12685, 2), (12685, 7, 7, 512))

## Merging & Splitting

- merging feature data & target data

In [13]:
# y_data = np.concatenate((y_train, y_test))

In [14]:
# X_data = np.concatenate((X_train, X_test))

In [15]:
# y_data.shape, X_data.shape

- splitting data (15% test)

In [16]:
# X_train, X_test, y_train, y_test = train_test_split(
#                                     X_data, y_data, test_size=0.15, random_state=42)

In [17]:
# y_train.shape, X_train.shape, y_test.shape, X_test.shape

## Utils Function

In [18]:
import json

def readJson_config(Path, Name, Key):
    with open(Path + Name) as json_config:
        json_object = json.load(json_config)

    return json_object[Key]


def writeJson_config(Path, Name, Data, append):
    mode = 'a+' if append else 'w'
    full_path = Path + Name

    with open(full_path, mode=mode) as json_config:
        json.dump(Data, json.load(json_config) if append else json_config)
    
    return 'success' 


## Building Convolutional Neural Network

- Import Keras library

In [19]:
from keras.models import Sequential
from keras.layers import Dense, Conv2D, MaxPool2D, Flatten, Dropout
from keras.layers import Input
from keras.layers.normalization import BatchNormalization
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.utils import plot_model

import keras

- Buat CNN Model dengan aritektur network : 
`CONV-POOL-CONV-POOL-CONV-POOL-FC`
- CONV : 1D Convolutional Layer
- POOL : MAX Pooling Layer
- FC   : Dense Layer + Activation

In [20]:
def cnn_model(input_shape):
    
    model = Sequential()
    
    model.add(Conv2D(filters=128,
                     kernel_size=3,
                     activation='relu',
                     input_shape=input_shape))
    model.add(BatchNormalization())
    model.add(Conv2D(filters=64,
                     kernel_size=3,
                     activation='relu',
                     input_shape=input_shape))
    model.add(BatchNormalization())
    model.add(MaxPool2D(pool_size=2,
                        strides=2,
                        padding='same'))
    

    # Fully Connected layer (FC)
    model.add(Flatten())
    model.add(Dense(128, 
                    activation='relu'))
    model.add(Dense(64, 
                    activation='relu'))
    model.add(Dense(2, 
                    activation='softmax'))
              
    model.summary()
    model.compile(optimizer='adam', 
                  loss='categorical_crossentropy',
                  metrics = ['accuracy'])

    return model

- sekarang kita akan melakukan proses training model dengan memanfaatkan `.fit()` pada model yang kita buat diatas.
- selain itu kita gunakan juka teknik `EarlyStoping()` untuk menghentikan proses training jika terjadi divergensi pada validation data yang diakibatkan oleh overfitting. 
- pada `EarlyStoping()` kita gunakan parmeter `patience=8` yang artinya jika proses training untuk 8 epoch tidak terjadi peningkatan maka hentikan proses training.

In [21]:
def check_model(model_, x, y, x_test, y_test, epochs_, batch_size_):
    callbacks = [EarlyStopping(monitor='val_loss', patience=3),
                 ModelCheckpoint(filepath='cnn_classif_best_model.h5', monitor='val_loss', save_best_only=True)]

    hist = model_.fit(x, 
                      y,
                      epochs=epochs_,
                      callbacks=callbacks, 
                      batch_size=batch_size_,
                      shuffle=True,
                      #validation_split=0.15)
                      validation_data=(x_test, y_test))
    #model_.load_weights('cnn_classif_best_model.h5')
    return hist 

### Train Model CNN

- jalankan proses training dengan `EPOCH` sebanyak 16 dan `BATCH_SIZE` sebesar 32

In [22]:
input_shape = X_train[0].shape

model = cnn_model(input_shape)

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_1 (Conv2D)            (None, 5, 5, 128)         589952    
_________________________________________________________________
batch_normalization_1 (Batch (None, 5, 5, 128)         512       
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 3, 3, 64)          73792     
_________________________________________________________________
batch_normalization_2 (Batch (None, 3, 3, 64)          256       
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 2, 2, 64)          0         
_________________________________________________________________
flatten_1 (Flatten)          (None, 256)               0         
_________________________________________________________________
dense_1 (Dense)              (None, 128)               32896     
__________

- save model arhitecture

In [23]:
with open(experiment_folder + experiment_name +
              '/model_summary_%s.txt' % feature_type, 'w') as f:
    model.summary(print_fn=lambda x: f.write(x + '\n'))

- train model

In [ ]:
EPOCHS = 16
BATCH_SIZE = 32

history=check_model(model, X_train, y_train, X_test, y_test, EPOCHS, BATCH_SIZE)

Train on 32620 samples, validate on 12685 samples
Epoch 1/16
   96/32620 [..............................] - ETA: 3:11:06 - loss: 0.6572 - acc: 0.6250

C:\Users\yunus\Anaconda3\envs\GPU_ENV\lib\site-packages\keras\callbacks.py:122: UserWarning: Method on_batch_end() is slow compared to the batch update (0.409233). Check your callbacks.
  % delta_t_median)


32620/32620 [==============================] - 121s 4ms/step - loss: 0.2506 - acc: 0.8979 - val_loss: 0.2503 - val_acc: 0.8974
Epoch 2/16
32620/32620 [==============================] - 90s 3ms/step - loss: 0.1673 - acc: 0.9332 - val_loss: 0.1950 - val_acc: 0.9187
Epoch 3/16
32620/32620 [==============================] - 87s 3ms/step - loss: 0.1325 - acc: 0.9482 - val_loss: 0.1958 - val_acc: 0.9302
Epoch 4/16
32620/32620 [==============================] - 93s 3ms/step - loss: 0.1060 - acc: 0.9593 - val_loss: 0.1739 - val_acc: 0.9359
Epoch 5/16
32608/32620 [============================>.] - ETA: 0s - loss: 0.0870 - acc: 0.9673- ETA: 2s - loss: 0.0869 - acc: 0.96 - ETA

- Save model

In [ ]:
model.save(experiment_folder + experiment_name + "/CNN_Classification_model_%s.h5" % feature_type)

- save model training history (log)

In [ ]:
pd.DataFrame.from_dict(history.history).to_csv( experiment_folder + experiment_name +
                                               '/history_train_CNN_feature_%s.csv' % feature_type, index=False)

# Evaluate Model

- Plot Accuracy vs Epochs
- Plot Loss vs Epochs
- Plot Confusion Matrix

In [ ]:
def evaluate_model(history):
    
    fig1, ax_acc = plt.subplots()
    plt.plot(history.history['acc'])
    plt.plot(history.history['val_acc'])
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Model - Accuracy')
    plt.legend(['Training', 'Validation'], loc='lower right')
    plt.grid()
    plt.show()
    fig1.savefig(experiment_folder + experiment_name +
               "/plot-accuracy-%s.png" % feature_type)
    
    fig2, ax_loss = plt.subplots()
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Model- Loss')
    plt.legend(['Training', 'Validation'], loc='upper right')
    plt.grid()
    plt.show()
    fig2.savefig(experiment_folder + experiment_name +
               "/plot-loss-%s.png" % feature_type)

evaluate_model(history)    

- Plot Confusion Matrix

In [ ]:
import itertools
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(5, 5))
    
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    
    plt.savefig(experiment_folder + experiment_name +
               "/plot-confusion-matrix-%s.png" % feature_type)
    
    plt.show()

In [ ]:
# predict test data
y_pred=model.predict(X_test)

In [ ]:
# Compute confusion matrix
cnf_matrix = confusion_matrix(y_test.argmax(axis=1), y_pred.argmax(axis=1))
np.set_printoptions(precision=2)


# Plot non-normalized confusion matrix
plot_confusion_matrix(cnf_matrix, classes=['AF', 'N'],normalize=True,
                      title='Confusion matrix, with normalization')

- dari hasil plot confusion matrix, dapat dilihat tiap kelas memiliki banyak TRUE POSITIVE predicted data
- semakin gelap kebiruan menunjukan banyaknya hasil predicted label untuk true label tersebut

- Plot Classification Report

In [ ]:
print(classification_report(y_test.argmax(axis=1), 
                            y_pred.argmax(axis=1), 
                            target_names=['AF', 'N']))

- Jika kita lihat, nilai report untuk seluruh klas juga bagus, 
- Nilai recall dan precission juga tinggi, menunjukan model mampu memprediksi data dengan baik untuk seluruh data pada sclass tersebut 

- save report

In [ ]:
report_dict = classification_report(y_test.argmax(axis=1), 
                            y_pred.argmax(axis=1), 
                            target_names=['AF', 'N'],
                            output_dict=True)

writeJson_config(experiment_folder + experiment_name + "/", 
                 "report-%s.json" % feature_type, report_dict, False)

- save model spec

In [ ]:
model_spec_dict = {}
model_spec_dict['train_size'] = [y_train.shape, X_train.shape]
model_spec_dict['test_size'] =  [y_test.shape, X_test.shape]
model_spec_dict['epoch'] =  EPOCHS
model_spec_dict['batch_size'] =  BATCH_SIZE

writeJson_config(experiment_folder + experiment_name + "/", 
                 "model-spec-%s.json" % feature_type, model_spec_dict, False)

- Update Experiment Header

In [ ]:
with open(experiment_folder + 
              '/experiment_header.txt', 'a') as f:
    f.write("Experiment Name \t: %s \n" % experiment_name)
    f.write("Accuracy \t\t: %.4f\n\n\n" % report_dict['accuracy'])

In [ ]:
if report_dict['accuracy'] >= threshold_acc:
    import shutil
    
    shutil.copy(experiment_folder + experiment_name + "plot-accuracy-vgg16.png", "5. plot-accuracy-vgg16.png")
    shutil.copy(experiment_folder + experiment_name + "plot-loss-vgg16.png", "5. plot-loss-vgg16.png")
    shutil.copy(experiment_folder + experiment_name + "plot-confusion-matrix-vgg16.png", "5. plot-confusion-matrix-vgg16.png")
    shutil.copy(experiment_folder + experiment_name + "history_train_CNN_feature_vgg16.csv", "history_train_CNN_feature_vgg16.csv")
    shutil.copy(experiment_folder + experiment_name + "report-vgg16.json", "classification-report-vgg16.json")
    shutil.copy(experiment_folder + experiment_name + "CNN_Classification_model_vgg16.h5", "CNN_Classification_model_vgg16.h5")
    print("[INFO] success move best result to main dir!")
else :
    print("[INFO] accuracy %.4f, is under threshold !" % report_dict['accuracy'])